### Create Gaussian Process around dataset

In [1]:
from sklearn.model_selection import train_test_split
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from notebooks.icsoc.globals import DATASET_LOG_SPLITS, \
    PATH_METRICS_ICSOC_EXPLORE, OVERALL_DATASET_USAGE

plt.style.use('default')

from agent.components.GaussianProcess import GASK
from agent.components.commons import ServiceType, theoretical_param_bounds, ServiceVar, FIG_SIZE_DEFAULT
from agent.components.commons import SloSet

largest_split = DATASET_LOG_SPLITS[-1]
_df_explore = pd.read_csv(PATH_METRICS_ICSOC_EXPLORE)
_df_total = _df_explore.sample(frac=OVERALL_DATASET_USAGE, random_state=35)
train_df, test_df = train_test_split(_df_total, test_size=0.20, random_state=35)

services = [ServiceType.QR, ServiceType.CV, ServiceType.PC]
slos = [SloSet.DEFAULT, SloSet.HIGH_PERF, SloSet.LOW_COST, SloSet.HIGH_QUALITY]

#### Create Gaussian Processes with certain shares of the exploration data set

In [2]:
import plotly.io as pio

pio.renderers.default = 'browser'

lml_history = []

gp_list = []
lml_history = []

for data_ratio in DATASET_LOG_SPLITS:
    gp_all_services = {}
    lml_all_service = []

    for s in services:
        # Initialize and train GP model
        draw_figures = data_ratio in [0.05, 0.5]
        _gp = GASK(s, create_figures=draw_figures, display_figures=draw_figures)
        _gp.init_model(train_df, data_density=data_ratio)

        _lml = _gp.get_model_lml(s, "max_tp")
        lml_scaled = _lml / data_ratio
        lml_all_service.append(lml_scaled)
        gp_all_services[s] = _gp

    lml_history.append(lml_all_service)
    gp_list.append(gp_all_services)

    print(f"Finished fitting GP with {data_ratio}% data ratio")



INFO:multiscale:draw_3d_gp_plot took 5136 ms to execute
INFO:multiscale:draw_3d_gp_plot took 242 ms to execute
INFO:multiscale:draw_3d_gp_plot took 420 ms to execute


Finished fitting GP with 0.05% data ratio
Finished fitting GP with 0.1% data ratio


INFO:multiscale:draw_3d_gp_plot took 714 ms to execute
INFO:multiscale:draw_3d_gp_plot took 1251 ms to execute
INFO:multiscale:draw_3d_gp_plot took 466 ms to execute


Finished fitting GP with 0.25% data ratio
Finished fitting GP with 0.5% data ratio
Finished fitting GP with 1.0% data ratio


### Create LML Figure

In [ ]:
data_ratios = np.arange(1, ITERATE_THROUGH_X_PARTS + 1) / SPLIT_DATA_INTO_X_PARTS
plt.figure(figsize=FIG_SIZE_DEFAULT)

# Service labels matching your 'services' list order
labels = ['QR Detector', 'CV Analyzer', 'PC Visualizer']
colors = ['#d62728', '#1f77b4', '#2ca02c']  # Red, Blue, Green

lml_array = np.array(lml_history)

for i in range(lml_array.shape[1]):
    plt.plot(data_ratios * (SPLIT_DATA_INTO_X_PARTS / ITERATE_THROUGH_X_PARTS), lml_array[:, i],
             marker='o', markersize=4, linewidth=2,
             label=labels[i], color=colors[i])

# 3. Formatting
abs_samples = int(len(_df_explore) * (ITERATE_THROUGH_X_PARTS / SPLIT_DATA_INTO_X_PARTS))
plt.xlabel(f'Ratio of Training Data ({int(abs_samples / 3)} Samples)', fontsize=12)
plt.ylabel('Scaled LML / # Samples', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title="Service Type")
plt.show()

In [ ]:
RUN_HEATMAP = False

if RUN_HEATMAP:

    from agent.components.GaussianProcess import get_empirical_variable_bounds
    from utils import visualize_ndarray
    from agent.components.commons import ServiceVar
    from typing import Dict
    from agent.components.Optimizer_v2 import VersatileMapElites


    def extract_pfo_for_SLOs(gp_service, slos: Dict[ServiceVar, float], slo_type: str, simple_bounds):
        v_me = VersatileMapElites(gp_service.s_type, bins=8)

        #  I'm getting the black cells because they are not explored.
        #  What I can do is force all cells to be explored at least once,
        #  or just run gradient descent for each cell multiple (like 5) times.
        v_me.run_search(slos, gp_service, simple_bounds, iterations=2000)
        visualize_ndarray(v_me.fitness_table, gp_service.s_type.value + "_" + slo_type, cmap="viridis")

        # (2) Here I get n best solutions
        # Get n solutions that are high-performing but far apart
        # diverse_set = v_me.get_diverse_set(n_solutions=10, versatility=0.1)
        # return diverse_set
        # print("\n".join(f"Versatile Candidate: {x}" for x in diverse_set))


    final_empirical_bounds = get_empirical_variable_bounds(df_explore_preprocessed)[ServiceType.QR]
    simple_param_bounds = final_empirical_bounds.copy()
    del simple_param_bounds[ServiceVar.PERFORMANCE]
    simple_param_bounds = list(simple_param_bounds.values())

    candidate_solutions = []
    # (1) Here I give it increasingly more training data
    for i in range(tested_range):
        candidates = extract_pfo_for_SLOs(gp_list[i][ServiceType.QR], SloSet.DEFAULT.value, "DEFAULT",
                                          simple_param_bounds)
        candidate_solutions.append(candidates)

#### Find optimal parameter configs for each SLO x Service combination after seeing certain data shares

In [ ]:
from agent.components.Optimizer_v2 import run_optimizer_multi

solution_history = []

for i in range(tested_range):
    for q in slos:
        for s in services:
            data_ratio = (i + 1) / tested_range
            gp = gp_list[i][s]

            # Run optimizer to find the best configuration
            solutions = run_optimizer_multi(s, q.value, gp, theoretical_param_bounds[s], runs=25)
            fitness, config = max(solutions, key=lambda x: x[0])

            # Predict performance (mu, sigma) for the chosen configuration
            x_state = {ServiceVar.COST: config[0], ServiceVar.QUALITY: config[1]}
            x_state = x_state | ({ServiceVar.MODEL: config[2]} if s == ServiceType.CV else {})
            mu, sigma = gp.predict(s, "max_tp", x_state)

            # Store everything needed for the next block
            # We include empirical_var_bounds here as it changes per iteration
            solution_history.append({
                'data_rate': data_ratio,
                'rep': None,
                'service_type': s.value,
                'slo_set': q.name,
                'p_fitness': fitness,
                # 'config': config,
                'dist': (mu, sigma),
                'cores': x_state[ServiceVar.COST],
                'data_quality': x_state[ServiceVar.QUALITY],
                'model_size': x_state[ServiceVar.MODEL] if s == ServiceType.CV else -1,
                # 'x_state': x_state,
            })

            print(f"Optimal fitness for {s.value} and {q.name} with {data_ratio * 100}% data: {fitness}")

#### Export to candidate solution script, with each config repeated x times

In [ ]:
repeatable_data = []
runs_per_config = 50

# Iterate through the list in steps of 3 (the size of your service triple)
for i in range(0, len(solution_history), 3):
    # Extract the current triple of rows
    triple = solution_history[i: i + 3]

    # Repeat this specific triple for the number of runs
    for run_idx in range(runs_per_config):
        for row in triple:
            new_row = row.copy()
            new_row['rep'] = run_idx + 1
            repeatable_data.append(new_row)

df_candidates = pd.DataFrame(repeatable_data)
df_candidates.to_csv(f'../statics/candidates/candidate_solutions_{tested_range}_{data_splits}_{runs_per_config}.csv',
                     index=False)